# SVAMITVA High-mIoU Segmentation — Dual V100 (2×32GB)
**Target:** 70–80% mIoU | **Encoder:** SegFormer-B5 | **GPUs:** 0+1 via DataParallel

### Key improvements over baseline
- `mit_b5` encoder (+3–5% mIoU vs b3)
- Train at **512px crops**, infer with **sliding window** at 1024px
- Checkpoint saved on **mIoU** (not pixel accuracy)
- Aggressive **Dice-heavy loss** (0.25 CE / 0.55 Dice / 0.20 Focal)
- **Road class restored** (was incorrectly dropped)
- **Inverse-frequency tile sampler** for rare class oversampling
- **TTA** (flip×3) at inference
- Safe `GradScaler` init to prevent epoch-1 NaN crash


In [1]:
# ── CELL 1: ENV + IMPORTS ────────────────────────────────────────────────────
import os, gc, glob, json, random, time
import subprocess, sys

# Must be set BEFORE torch import
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'
os.environ['CUDA_VISIBLE_DEVICES']    = '0,1'   # both V100s

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'segmentation-models-pytorch', 'albumentations',
    'rasterio', 'scikit-learn', 'timm'], capture_output=True)

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import rasterio
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

gc.collect()
torch.cuda.empty_cache()

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

n_gpus  = torch.cuda.device_count()
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU   = max(n_gpus, 1)

print('=' * 70)
print(f'  GPUs available : {n_gpus}')
for i in range(n_gpus):
    f, t = torch.cuda.mem_get_info(i)
    print(f'  cuda:{i}  {torch.cuda.get_device_name(i)}  '
          f'free={f/1e9:.1f}GB / total={t/1e9:.1f}GB')
print('=' * 70)


  GPUs available : 2
  cuda:0  Tesla V100-PCIE-32GB  free=33.7GB / total=34.1GB
  cuda:1  Tesla V100-PCIE-32GB  free=33.7GB / total=34.1GB


In [2]:
# ── CELL 2: PATHS + CLASS SCHEMA ─────────────────────────────────────────────
BASE      = '/home/jupyter-228w1a1286/dinesh-data/hackthonndata'
CKPT_DIR  = f'{BASE}/CHECKPOINTS_MIOU'
OUT_DIR   = f'{BASE}/FINAL_OUTPUTS'
IMG_DIR   = f'{BASE}/trainingdata/DATASET_1024/images'
MASK_DIR  = f'{BASE}/trainingdata/DATASET_1024/masks'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

# ── 5-class schema (road restored, water_body_line dropped as line annotation)
# Original → New
#  0 background       → 0  background
#  1 built_up_area    → 1  built_up_area    (24.1%)
#  2 road             → 2  road             (0.46% — restored!)
#  3 road_centre_line → 3  road_centre_line (6.9%)
#  4 water_body       → 4  water_body       (2.9%)
#  5 water_body_line  → 0  background       (line anno, too thin)
#  6 waterbody_point  → 0  background       (point anno)
#  7 utility          → 0  background       (sparse)
#  8 utility_poly     → 0  background       (sparse)
REMAP_ARRAY = np.array([0, 1, 2, 3, 4, 0, 0, 0, 0], dtype=np.int64)

NUM_CLASSES    = 5
CLASS_NAMES    = ['background', 'built_up_area', 'road', 'road_centre_line', 'water_body']
ACTIVE_CLASSES = [1, 2, 3, 4]   # exclude background from mIoU

COLOR_MAP = np.array([
    [0,   0,   0  ],  # 0 background
    [255, 0,   0  ],  # 1 built_up_area
    [255, 165, 0  ],  # 2 road
    [255, 255, 0  ],  # 3 road_centre_line
    [0,   0,   255],  # 4 water_body
], dtype=np.uint8)

# Pixel counts after remapping
PIXEL_COUNTS = np.array([
    1_764_710_366 + 75_806_214 + 543_668 + 44_707_486 + 1_650_674,  # 0 bg
    693_864_908,  # 1 built_up_area
     13_185_924,  # 2 road
    199_538_875,  # 3 road_centre_line
     83_284_429,  # 4 water_body
], dtype=np.float64)

print('Class distribution after remapping:')
total = PIXEL_COUNTS.sum()
for i, (n, c) in enumerate(zip(CLASS_NAMES, PIXEL_COUNTS)):
    print(f'  {i} {n:22s}: {int(c):>14,}  ({100*c/total:.3f}%)')


Class distribution after remapping:
  0 background            :  1,887,418,408  (65.597%)
  1 built_up_area         :    693,864,908  (24.115%)
  2 road                  :     13,185,924  (0.458%)
  3 road_centre_line      :    199,538,875  (6.935%)
  4 water_body            :     83,284,429  (2.895%)


In [3]:
# ── CELL 3: CONFIG ────────────────────────────────────────────────────────────
TRAIN_TILE = 512          # train on 512px crops (4× more diversity)
PER_GPU    = 8            # 512px on V100-32GB → safe at 8/GPU
EFF_BS     = PER_GPU * N_GPU   # 16 with 2 GPUs

CFG = dict(
    img_dir         = IMG_DIR,
    mask_dir        = MASK_DIR,
    save_dir        = CKPT_DIR,
    num_classes     = NUM_CLASSES,
    train_tile      = TRAIN_TILE,
    infer_tile      = 1024,
    infer_overlap   = 128,
    encoder         = 'mit_b5',           # upgraded from b3
    encoder_weights = 'imagenet',
    batch_size      = EFF_BS,
    accum_steps     = 1,
    num_epochs      = 80,
    lr              = 4e-5,
    weight_decay    = 1e-2,
    val_split       = 0.15,
    num_workers     = 4,
    ce_weight       = 0.25,   # reduced — background domination
    dice_weight     = 0.55,   # increased — directly optimises IoU
    focal_weight    = 0.20,
    patience        = 15,
    min_delta       = 5e-4,
    resume          = False,
    resume_ckpt     = f'{CKPT_DIR}/best_model.pth',
)

print(f'Encoder        : {CFG["encoder"]}')
print(f'Train tile     : {TRAIN_TILE}px  (infer: {CFG["infer_tile"]}px sliding window)')
print(f'Batch / GPU    : {PER_GPU}')
print(f'Effective batch: {EFF_BS}')
print(f'Epochs         : {CFG["num_epochs"]}')
print(f'Loss weights   : CE={CFG["ce_weight"]} Dice={CFG["dice_weight"]} Focal={CFG["focal_weight"]}')


Encoder        : mit_b5
Train tile     : 512px  (infer: 1024px sliding window)
Batch / GPU    : 8
Effective batch: 16
Epochs         : 80
Loss weights   : CE=0.25 Dice=0.55 Focal=0.2


In [4]:
# ── CELL 4: DATASET ───────────────────────────────────────────────────────────
class SVAMITVADataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.imgs      = img_paths
        self.masks     = mask_paths
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        with rasterio.open(self.imgs[idx]) as s:
            img  = s.read().astype(np.float32) / 255.0
            img  = np.transpose(img, (1, 2, 0))   # H W C
        with rasterio.open(self.masks[idx]) as s:
            mask = s.read(1).astype(np.int64)

        mask = REMAP_ARRAY[np.clip(mask, 0, 8)]

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']
            mask = aug['mask'].long()
        else:
            img  = torch.from_numpy(img.transpose(2, 0, 1)).float()
            mask = torch.from_numpy(mask).long()
        return img, mask

    def get_sample_weight(self, idx):
        """Inverse-frequency weight — boosts tiles with rare classes."""
        with rasterio.open(self.masks[idx]) as s:
            mask = REMAP_ARRAY[np.clip(s.read(1).flatten(), 0, 8)]
        inv_freq = 1.0 / (PIXEL_COUNTS / PIXEL_COUNTS.sum())
        present  = np.unique(mask)
        active   = [c for c in present if c > 0]
        return float(max(inv_freq[c] for c in active)) if active else 1.0


# ── AUGMENTATIONS ─────────────────────────────────────────────────────────────
train_tf = A.Compose([
    A.RandomCrop(CFG['train_tile'], CFG['train_tile']),  # 512px crop from 1024px tile
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.2),
        A.OpticalDistortion(distort_limit=0.15),
    ], p=0.2),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        A.CLAHE(clip_limit=4.0),
    ], p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
    A.CoarseDropout(max_holes=4, max_height=48, max_width=48, fill_value=0, p=0.15),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.RandomCrop(CFG['train_tile'], CFG['train_tile']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


In [5]:
# ── CELL 5: DATALOADERS ───────────────────────────────────────────────────────
all_imgs  = sorted(glob.glob(f"{CFG['img_dir']}/*.tif"))
all_masks = sorted(glob.glob(f"{CFG['mask_dir']}/*.tif"))

assert len(all_imgs) > 0,               f'No tiles found in {CFG["img_dir"]}'
assert len(all_imgs) == len(all_masks), 'Img/mask count mismatch'

idx = list(range(len(all_imgs)))
tr_idx, va_idx = train_test_split(idx, test_size=CFG['val_split'], random_state=SEED)

tr_imgs  = [all_imgs[i]  for i in tr_idx]
tr_masks = [all_masks[i] for i in tr_idx]
va_imgs  = [all_imgs[i]  for i in va_idx]
va_masks = [all_masks[i] for i in va_idx]

train_ds = SVAMITVADataset(tr_imgs, tr_masks, train_tf)
val_ds   = SVAMITVADataset(va_imgs, va_masks, val_tf)

# Inverse-frequency sample weights (cached)
wcache = os.path.join(CFG['save_dir'], 'sw_miou.json')
if os.path.exists(wcache):
    sw = json.load(open(wcache))
    if len(sw) != len(train_ds): sw = None
    else: print('Loaded cached sample weights')
else:
    sw = None

if sw is None:
    print('Computing sample weights (inverse-frequency)...')
    sw = [train_ds.get_sample_weight(i)
          for i in tqdm(range(len(train_ds)), desc='weights')]
    json.dump(sw, open(wcache, 'w'))

sampler = WeightedRandomSampler(sw, len(train_ds), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size         = CFG['batch_size'],
    sampler            = sampler,
    num_workers        = CFG['num_workers'],
    pin_memory         = True,
    prefetch_factor    = 2,
    persistent_workers = False,
    drop_last          = True,
)
val_loader = DataLoader(
    val_ds,
    batch_size         = CFG['batch_size'],
    shuffle            = False,
    num_workers        = CFG['num_workers'],
    pin_memory         = True,
    prefetch_factor    = 2,
    persistent_workers = False,
)

print(f'Total: {len(all_imgs)} | Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Steps/epoch: {len(train_loader)}  (batch={CFG["batch_size"]})')


Loaded cached sample weights
Total: 2744 | Train: 2332 | Val: 412
Steps/epoch: 145  (batch=16)


In [6]:
# ── CELL 6: MODEL ─────────────────────────────────────────────────────────────
gc.collect(); torch.cuda.empty_cache()

base = smp.Segformer(
    encoder_name    = CFG['encoder'],
    encoder_weights = CFG['encoder_weights'],
    in_channels     = 3,
    classes         = CFG['num_classes'],
    activation      = None,
).to(DEVICE)

# Gradient checkpointing — saves ~40% VRAM on mit_b5
if hasattr(base.encoder, 'set_grad_checkpointing'):
    base.encoder.set_grad_checkpointing(True)
    print('Grad checkpointing: via set_grad_checkpointing')
else:
    for m in base.encoder.modules():
        if hasattr(m, 'gradient_checkpointing'):
            m.gradient_checkpointing = True
    print('Grad checkpointing: via module flags')

# DataParallel — no torch.compile (incompatible with SMP)
if N_GPU > 1:
    model = nn.DataParallel(base, device_ids=list(range(N_GPU)))
    print(f'DataParallel: {N_GPU} GPUs')
else:
    model = base
    print('Single GPU')

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {n_params:,}')
for i in range(N_GPU):
    f, t = torch.cuda.mem_get_info(i)
    print(f'  cuda:{i} free: {f/1e9:.1f} GB / {t/1e9:.1f} GB')


Grad checkpointing: via module flags
DataParallel: 2 GPUs
Params: 81,970,117
  cuda:0 free: 33.4 GB / 34.1 GB
  cuda:1 free: 33.7 GB / 34.1 GB


In [7]:
# ── CELL 7: LOSS + OPTIM ──────────────────────────────────────────────────────
raw_w = 1.0 / np.log1p(PIXEL_COUNTS)
raw_w = np.clip(raw_w / raw_w.mean(), 0, 15.0)
cw    = torch.tensor(raw_w, dtype=torch.float32).to(DEVICE)
print('Class weights:', [f'{w:.3f}' for w in raw_w])

ce_fn    = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
dice_fn  = smp.losses.DiceLoss('multiclass', classes=ACTIVE_CLASSES, smooth=1.0)
focal_fn = smp.losses.FocalLoss('multiclass', gamma=2.0)   # 2.0 safer than 3.0

def criterion(out, mask):
    return (CFG['ce_weight']    * ce_fn(out, mask) +
            CFG['dice_weight']  * dice_fn(out, mask) +
            CFG['focal_weight'] * focal_fn(out, mask))

enc_p = [p for n,p in unwrap(model).named_parameters() if 'encoder' in n]
dec_p = [p for n,p in unwrap(model).named_parameters() if 'encoder' not in n]

opt = torch.optim.AdamW([
    {'params': enc_p, 'lr': CFG['lr'],     'weight_decay': CFG['weight_decay']},
    {'params': dec_p, 'lr': CFG['lr']*10,  'weight_decay': CFG['weight_decay']},
])

def warmup_cosine(ep, warmup=5, total=80):
    if ep < warmup: return (ep+1)/warmup
    return 0.5*(1+np.cos(np.pi*(ep-warmup)/(total-warmup)))

sched  = torch.optim.lr_scheduler.LambdaLR(
    opt, lr_lambda=lambda e: warmup_cosine(e, 5, CFG['num_epochs']))

# Safe init_scale prevents NaN crash on epoch 1
scaler = GradScaler('cuda', init_scale=2**12, growth_interval=200)
print('Loss / optim ready')


Class weights: ['0.886', '0.930', '1.155', '0.991', '1.038']
Loss / optim ready


In [8]:
# ── CELL 8: RESUME (optional) ─────────────────────────────────────────────────
best_miou = 0.0
start_ep  = 0

if CFG['resume'] and os.path.exists(CFG['resume_ckpt']):
    ck = torch.load(CFG['resume_ckpt'], map_location=DEVICE, weights_only=False)
    st = {k.replace('_orig_mod.', ''): v for k, v in ck['model_state'].items()}
    unwrap(model).load_state_dict(st, strict=False)
    best_miou = ck.get('val_miou', 0.0)
    start_ep  = ck.get('epoch', 0)
    print(f'Resumed from epoch {start_ep}  best_mIoU={best_miou:.4f}')
else:
    print('Training from scratch')


Training from scratch


In [9]:
# ── CELL 9: METRICS ───────────────────────────────────────────────────────────
def compute_metrics(preds, masks):
    """Pixel accuracy + per-class IoU dict + mIoU."""
    acc  = (preds == masks).float().mean().item()
    ious = {}
    for c in ACTIVE_CLASSES:
        inter = ((preds==c) & (masks==c)).sum().item()
        union = ((preds==c) | (masks==c)).sum().item()
        if union > 0: ious[c] = inter / union
    miou = float(np.mean(list(ious.values()))) if ious else 0.0
    return acc, ious, miou

print('Metrics helper ready')


Metrics helper ready


In [ ]:
# ── CELL 10: TRAINING LOOP ────────────────────────────────────────────────────
hist        = dict(tl=[], vl=[], acc=[], miou=[])
patience_ct = 0
nan_total   = 0

def mem_str():
    return ' | '.join(
        f'GPU{i}:{torch.cuda.max_memory_allocated(i)/1e9:.1f}GB'
        for i in range(N_GPU)
    )

print('\n' + '═'*70)
print(f'  {CFG["encoder"].upper()} | {N_GPU}×GPU | '
      f'train={CFG["train_tile"]}px crop | batch={CFG["batch_size"]}')
print(f'  Classes : {CLASS_NAMES}')
print(f'  Target  : mIoU 70–80%')
print('═'*70 + '\n')

for epoch in range(start_ep, CFG['num_epochs']):
    for i in range(N_GPU): torch.cuda.reset_peak_memory_stats(i)
    gc.collect(); torch.cuda.empty_cache()
    t0 = time.time()

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    tl, ts, ne = 0.0, 0, 0
    opt.zero_grad()

    for step, (imgs, masks) in enumerate(tqdm(
            train_loader, desc=f'Ep{epoch+1:03d} train', leave=False, ncols=90)):

        imgs  = imgs.to(DEVICE,  non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast('cuda'):
            out  = model(imgs)
            out  = torch.clamp(out, -50, 50)   # guard ±inf logits
            loss = criterion(out, masks)

        if torch.isnan(loss) or torch.isinf(loss):
            ne += 1
            del imgs, masks, out, loss
            opt.zero_grad(); torch.cuda.empty_cache(); continue

        scaler.scale(loss).backward()

        scaler.unscale_(opt)
        has_nan = any(
            p.grad is not None and
            (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
            for p in model.parameters()
        )
        if has_nan:
            ne += 1
            del imgs, masks, out, loss
            opt.zero_grad(); scaler.update()
            torch.cuda.empty_cache(); continue

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad()

        tl += loss.item(); ts += 1
        del imgs, masks, out, loss

        if step % 50 == 0:
            torch.cuda.empty_cache()

    sched.step()

    if ne > 0:
        nan_total += ne
        print(f'  ⚠️  {ne} NaN batches (total: {nan_total})')
    if ts == 0:
        print('  All NaN — stopping.'); break

    tl     /= ts
    tr_t    = time.time() - t0

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    model.eval()
    vl, vs   = 0.0, 0
    all_ious = {c: [] for c in ACTIVE_CLASSES}
    all_accs = []

    with torch.no_grad():
        for imgs, masks in tqdm(
                val_loader, desc=f'Ep{epoch+1:03d} val  ', leave=False, ncols=90):

            imgs  = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, masks)

            if torch.isnan(loss) or torch.isinf(loss):
                del imgs, masks, out, loss; continue

            preds = torch.argmax(out, 1)
            vl   += loss.item(); vs += 1

            acc, ious, _ = compute_metrics(preds, masks)
            all_accs.append(acc)
            for c, v in ious.items(): all_ious[c].append(v)

            del imgs, masks, out, loss, preds

    va_t  = time.time() - t0 - tr_t
    vl   /= max(vs, 1)
    pc    = {c: float(np.mean(v)) for c, v in all_ious.items() if v}
    miou  = float(np.mean(list(pc.values()))) if pc else 0.0
    acc   = float(np.mean(all_accs)) if all_accs else 0.0

    gc.collect(); torch.cuda.empty_cache()

    hist['tl'].append(tl);   hist['vl'].append(vl)
    hist['acc'].append(acc); hist['miou'].append(miou)

    # ── CHECKPOINT on mIoU (not accuracy) ─────────────────────────────────────
    improved = miou > best_miou + CFG['min_delta']
    if improved:
        best_miou   = miou; patience_ct = 0
        torch.save({
            'epoch':          epoch+1,
            'model_state':    unwrap(model).state_dict(),
            'optim_state':    opt.state_dict(),
            'val_miou':       miou,
            'val_acc':        acc,
            'cfg':            CFG,
            'class_names':    CLASS_NAMES,
            'active_classes': ACTIVE_CLASSES,
            'remap_array':    REMAP_ARRAY.tolist(),
            'color_map':      COLOR_MAP.tolist(),
        }, os.path.join(CFG['save_dir'], 'best_model.pth'))
        tag = f'  ✅ best mIoU={best_miou:.4f}'
    else:
        patience_ct += 1
        tag = f'  (pat {patience_ct}/{CFG["patience"]})'

    lr = opt.param_groups[0]['lr']
    print(f'Ep {epoch+1:03d}/{CFG["num_epochs"]} | '
          f'TL:{tl:.4f}({tr_t:.0f}s) VL:{vl:.4f}({va_t:.0f}s) | '
          f'Acc:{acc*100:.2f}% mIoU:{miou:.4f} | '
          f'LR:{lr:.1e} | {mem_str()}{tag}')

    if (epoch+1) % 10 == 0:
        print('  Per-class IoU:',
              {CLASS_NAMES[c]: f'{v:.3f}' for c, v in pc.items()})

    if patience_ct >= CFG['patience']:
        print(f'Early stop at epoch {epoch+1}'); break

print(f'\n✅ Best mIoU={best_miou:.4f}')



══════════════════════════════════════════════════════════════════════
  MIT_B5 | 2×GPU | train=512px crop | batch=16
  Classes : ['background', 'built_up_area', 'road', 'road_centre_line', 'water_body']
  Target  : mIoU 70–80%
══════════════════════════════════════════════════════════════════════



Ep001 train:   0%|                                                | 0/145 [00:00<?, ?it/s]

In [ ]:
# ── CELL 11: SAVE TRAINING CURVES ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(hist['tl'])+1)

axes[0].plot(ep, hist['tl'], label='Train'); axes[0].plot(ep, hist['vl'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, [v*100 for v in hist['acc']], color='blue')
axes[1].set_title('Val Pixel Accuracy (%)'); axes[1].grid(True, alpha=0.3)

axes[2].plot(ep, hist['miou'], color='green')
axes[2].axhline(0.70, color='red',    linestyle='--', label='70% target')
axes[2].axhline(0.80, color='orange', linestyle='--', label='80% target')
axes[2].set_title('Val mIoU'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(CFG['save_dir'], 'training_curves.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Curves saved → {out_path}')


In [ ]:
# ── CELL 12: LOAD BEST MODEL FOR INFERENCE ───────────────────────────────────
del model, opt, scaler, sched
gc.collect(); torch.cuda.empty_cache()
for i in range(N_GPU):
    f, _ = torch.cuda.mem_get_info(i)
    print(f'  cuda:{i} free after cleanup: {f/1e9:.1f} GB')

ck = torch.load(os.path.join(CFG['save_dir'], 'best_model.pth'),
                map_location=DEVICE, weights_only=False)

inf_model = smp.Segformer(
    encoder_name    = CFG['encoder'],
    encoder_weights = None,
    in_channels     = 3,
    classes         = CFG['num_classes'],
    activation      = None,
).to(DEVICE)

inf_model.load_state_dict(
    {k.replace('_orig_mod.', ''): v for k, v in ck['model_state'].items()},
    strict=True
)
inf_model.eval()
print(f'Best model loaded (epoch {ck["epoch"]}, mIoU={ck["val_miou"]:.4f})')


In [ ]:
# ── CELL 13: TTA + SLIDING WINDOW INFERENCE ──────────────────────────────────
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def tta_pred(tensor):
    """Average predictions over 4 augmentations: original + 3 flips."""
    with torch.no_grad():
        p0 = torch.softmax(inf_model(tensor),                               1)
        p1 = torch.flip(torch.softmax(inf_model(torch.flip(tensor, [3])),   1), [3])
        p2 = torch.flip(torch.softmax(inf_model(torch.flip(tensor, [2])),   1), [2])
        p3 = torch.rot90(torch.softmax(inf_model(torch.rot90(tensor, 1, [2,3])), 1), -1, [2,3])
    return (p0 + p1 + p2 + p3) / 4.0


def sliding_window_predict(tif_path, tile=512, overlap=128):
    """
    Sliding window inference on a full 1024px (or any size) GeoTIFF.
    Returns prediction mask (H, W) numpy array.
    """
    stride = tile - overlap

    with rasterio.open(tif_path) as src:
        img     = src.read().astype(np.float32) / 255.0   # C H W
        profile = src.profile
        transform_geo = src.transform
        crs     = src.crs

    _, H, W = img.shape
    accum  = np.zeros((NUM_CLASSES, H, W), dtype=np.float32)
    counts = np.zeros((H, W),              dtype=np.float32)

    ys = list(range(0, H - tile + 1, stride))
    xs = list(range(0, W - tile + 1, stride))
    if ys[-1] + tile < H: ys.append(H - tile)
    if xs[-1] + tile < W: xs.append(W - tile)

    for y in ys:
        for x in xs:
            patch = img[:, y:y+tile, x:x+tile]
            t = torch.from_numpy(patch).unsqueeze(0).to(DEVICE)
            t = (t - MEAN) / STD
            with autocast('cuda'):
                prob = tta_pred(t).squeeze(0).cpu().numpy()   # C H W
            accum[:, y:y+tile, x:x+tile] += prob
            counts[   y:y+tile, x:x+tile] += 1.0

    pred = np.argmax(accum / np.maximum(counts, 1e-6), axis=0).astype(np.uint8)
    return pred, profile, transform_geo, crs


print('TTA + sliding window inference helpers ready')


In [ ]:
# ── CELL 14: RUN INFERENCE ON TEST TILES ─────────────────────────────────────
import re
try:
    from shapely.geometry import shape, mapping
    from rasterio.features import shapes as rio_shapes
    import geopandas as gpd
    HAS_GEO = True
except ImportError:
    HAS_GEO = False
    print('geopandas not available — skipping vectorization')

TEST_TIFS = sorted(glob.glob(f'{BASE}/trainingdata/DATASET_1024/images/*.tif'))[:5]
# ↑ Change to your actual test path; using first 5 train tiles as demo

if not TEST_TIFS:
    print('No test TIFs found — update TEST_TIFS path')
else:
    for tif_path in tqdm(TEST_TIFS, desc='Inference'):
        vname = os.path.splitext(os.path.basename(tif_path))[0]
        vdir  = os.path.join(OUT_DIR, vname)
        os.makedirs(vdir, exist_ok=True)

        pred, profile, tgeo, crs = sliding_window_predict(
            tif_path,
            tile    = CFG['infer_tile'],
            overlap = CFG['infer_overlap'],
        )

        # Save prediction raster
        pred_path = os.path.join(vdir, f'{vname}_pred.tif')
        profile.update(count=1, dtype='uint8', compress='lzw')
        with rasterio.open(pred_path, 'w', **profile) as dst:
            dst.write(pred[np.newaxis, ...])

        # Save colorised RGB visualisation
        vis = COLOR_MAP[pred]
        vis_path = os.path.join(vdir, f'{vname}_vis.tif')
        vis_profile = profile.copy()
        vis_profile.update(count=3, dtype='uint8')
        with rasterio.open(vis_path, 'w', **vis_profile) as dst:
            dst.write(np.transpose(vis, (2, 0, 1)))

        # Vectorise to GeoPackage
        if HAS_GEO:
            feats = []
            for geom, val in rio_shapes(pred, transform=tgeo):
                cls_id = int(val)
                if cls_id == 0: continue
                feats.append({'geometry': shape(geom),
                              'class_id': cls_id,
                              'class_name': CLASS_NAMES[cls_id]})
            if feats:
                gdf = gpd.GeoDataFrame(feats, crs=crs)
                gpkg_path = os.path.join(vdir, f'{vname}.gpkg')
                gdf.to_file(gpkg_path, driver='GPKG')

        print(f'  ✅ {vname}')

    print(f'\nOutputs saved → {OUT_DIR}')


In [ ]:
# ── CELL 15: FINAL REPORT ────────────────────────────────────────────────────
from datetime import datetime
report = {
    'project':     'SVAMITVA AI Feature Extraction',
    'hackathon':   'IIT Tirupati Geospatial Intelligence Hackathon 2025-26',
    'generated':   datetime.now().isoformat(),
    'model':       f'SegFormer-{CFG["encoder"]}',
    'classes':     CLASS_NAMES,
    'best_miou':   round(best_miou, 4),
    'train_tile':  CFG['train_tile'],
    'infer_tile':  CFG['infer_tile'],
    'tta':         True,
    'loss':        {'ce': CFG['ce_weight'], 'dice': CFG['dice_weight'],
                    'focal': CFG['focal_weight']},
}
with open(os.path.join(OUT_DIR, 'report.json'), 'w') as f:
    json.dump(report, f, indent=2)

print('\n' + '='*60)
print('  FINAL REPORT')
print('='*60)
for k, v in report.items():
    print(f'  {k:15s}: {v}')
print('='*60)
